# Part 1.3: Class Imbalance

In [1]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler


c:\Users\Hzaab\Desktop\MLSD project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import mlflow
from sklearn.neighbors import KNeighborsClassifier

import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
print("Tracking URI:", mlflow.get_tracking_uri())
mlflow.utils.logging_utils.disable_logging()
mlflow.set_experiment("class-imbalance-comparison")


In [2]:
train=pd.read_csv("C:\\Users\\Hzaab\\Desktop\\MLSD project\\data\\raw\\train.csv")

train=train.drop(columns=["Unnamed: 0"])

log_cols = ["#follows", "#followers", "description length", "#posts"]
for col in log_cols:
    train[col] = np.log1p(train[col])

categorical_cols = ["profile pic", "name==username", "external URL", "private"]
numeric_columns = [col for col in train.columns if col not in categorical_cols and col != "fake"]

In [3]:
X=train.drop(columns=["fake"])
y=train["fake"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=10) #stratify is used to maintain the same proportion of classes in both training and validation sets


## Logistic regression Baseline

In [ ]:
#transformers for numeric and categorical features
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

#combine transformers into a preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)



In [ ]:
strategies = {
    "none": None,
    "class_weight_balanced": "balanced",
    "random_oversample": RandomOverSampler(random_state=10),
    "smote": SMOTE(random_state=10),
    "random_undersample": RandomUnderSampler(random_state=10),
}

In [ ]:
mlflow.set_experiment("class-imbalance-comparison")

categorical_cols = ['profile pic', 'name==username', 'external URL', 'private']

results = []

for strategy_name, strategy in strategies.items():
    run_name = f"logreg_{strategy_name}"

    with mlflow.start_run(run_name=run_name):
        #combine preprocessor and model into a pipeline, adding the imbalance strategy if specified 
        if strategy_name == "class_weight_balanced":
            model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=10)

            pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("model", model),
                ]
            )


        elif strategy is None:
            model = LogisticRegression(max_iter=2000, random_state=10)
            pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("model", model),
                ]
            )


        else:
            model = LogisticRegression(max_iter=2000, random_state=10)
            pipeline = ImbPipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("sampler", strategy),
                    ("model", model),
                ]
            )

        #fit the pipeline
        pipeline.fit(X_train, y_train)

        
        y_pred = pipeline.predict(X_val)
        y_prob = None
        y_prob = pipeline.predict_proba(X_val)
        y_prob = y_prob[:, 1]  #probabilities for the positive class


        accuracy = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, zero_division=0)
        recall = recall_score(y_val, y_pred, zero_division=0)
        f1 = f1_score(y_val, y_pred, zero_division=0)

        report = classification_report(y_val, y_pred, zero_division=0)
        cm = confusion_matrix(y_val, y_pred)

        mlflow.log_param("model", "LogisticRegression")
        mlflow.log_param("imbalance_strategy", strategy_name)
        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("random_state", 10)
        mlflow.log_param("n_numeric_cols", len(numeric_columns))
        mlflow.log_param("n_categorical_cols", len(categorical_cols))

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)

        report_path = f"classification_report_{strategy_name}.txt"
        with open(report_path, "w") as f:
            f.write(report)

        cm_path = f"confusion_matrix_{strategy_name}.txt"
        with open(cm_path, "w") as f:
            f.write(np.array2string(cm))

        mlflow.log_artifact(report_path)
        mlflow.log_artifact(cm_path)

        mlflow.sklearn.log_model(pipeline, name="model")

        results.append(
            {
                "strategy": strategy_name,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }
        )

        print(f"\n{strategy_name}):")
        print(report)
        print("Confusion matrix:")
        print(cm)

results_df = pd.DataFrame(results).sort_values(by=["f1", "recall", "precision"], ascending=False)

print("\nFinal comparison")
print(results_df)


=== none ===
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       400
           1       0.99      0.89      0.94       100

    accuracy                           0.98       500
   macro avg       0.98      0.94      0.96       500
weighted avg       0.98      0.98      0.98       500

Confusion matrix:
[[399   1]
 [ 11  89]]
🏃 View run logreg_none at: http://127.0.0.1:5000/#/experiments/1/runs/23d7792796294144b5c92066e0b0a5b7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

=== class_weight_balanced ===
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       400
           1       0.92      0.93      0.93       100

    accuracy                           0.97       500
   macro avg       0.95      0.96      0.95       500
weighted avg       0.97      0.97      0.97       500

Confusion matrix:
[[392   8]
 [  7  93]]
🏃 View run logreg_class_weight_balanced at: http://127.0

 We see Logistic regression performs very well even in the no class imbalance management case, except in recall, where all the "fixed" cases performed the same, from this we understand that this is an easy problem, to study the best management method further we test another ML algorithm, Random forest

## Random Forest Baseline

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

In [16]:

numeric_transformer_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_tree, numeric_columns),
        ("cat", categorical_transformer_tree, categorical_cols),
    ]
)

In [ ]:

rf_strategies = {
    "none": None,
    "class_weight_balanced": "balanced",
    "random_oversample": RandomOverSampler(random_state=10),
    "smote": SMOTE(random_state=10),
    "random_undersample": RandomUnderSampler(random_state=10),
}

rf_results = []

for strategy_name, strategy in rf_strategies.items():
    run_name = f"RF_{strategy_name}"

    with mlflow.start_run(run_name=run_name):
        if strategy_name == "class_weight_balanced":
            model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=10, n_jobs=-1)

            pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor_tree),
                    ("model", model),
                ]
            )

        elif strategy is None:
            model = RandomForestClassifier(n_estimators=200, random_state=10, n_jobs=-1 )

            pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor_tree),
                    ("model", model),
                ]
            )

        else:
            model = RandomForestClassifier(n_estimators=200, random_state=10, n_jobs=-1)

            pipeline = ImbPipeline(
                steps=[
                    ("preprocessor", preprocessor_tree),
                    ("sampler", strategy),
                    ("model", model),
                ]
            )

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_val)
        y_prob = None
        y_prob = pipeline.predict_proba(X_val)[:, 1]

        accuracy = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, zero_division=0)
        recall = recall_score(y_val, y_pred, zero_division=0)
        f1 = f1_score(y_val, y_pred, zero_division=0)

        report = classification_report(y_val, y_pred, zero_division=0)
        cm = confusion_matrix(y_val, y_pred)

        mlflow.log_param("model", "RandomForestClassifier")
        mlflow.log_param("imbalance_strategy", strategy_name)
        mlflow.log_param("n_estimators", 200)
        mlflow.log_param("random_state", 42)
        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("n_numeric_cols", len(numeric_columns))
        mlflow.log_param("n_categorical_cols", len(categorical_cols))

        if strategy_name == "class_weight_balanced":
            mlflow.log_param("class_weight", "balanced")

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)

        report_path = f"rf_classification_report_{strategy_name}.txt"
        with open(report_path, "w") as f:
            f.write(report)

        cm_path = f"rf_confusion_matrix_{strategy_name}.txt"
        with open(cm_path, "w") as f:
            f.write(np.array2string(cm))

        mlflow.log_artifact(report_path)
        mlflow.log_artifact(cm_path)

        mlflow.sklearn.log_model(pipeline, name="model")

        rf_results.append(
            {
                "model": "RandomForestClassifier",
                "strategy": strategy_name,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }
        )

        print(f"\n RF {strategy_name}:")
        print(report)
        print("Confusion matrix:")
        print(cm)

rf_results_df = pd.DataFrame(rf_results).sort_values(
    by=["f1", "recall", "precision"],
    ascending=False,
)

print("\n=== RF Final comparison ===")
print(rf_results_df)


=== RF none ===
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       400
           1       1.00      0.91      0.95       100

    accuracy                           0.98       500
   macro avg       0.99      0.96      0.97       500
weighted avg       0.98      0.98      0.98       500

Confusion matrix:
[[400   0]
 [  9  91]]
🏃 View run RF_none at: http://127.0.0.1:5000/#/experiments/1/runs/82c499e763274d81b2fad9decc7b6861
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

=== RF class_weight_balanced ===
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       400
           1       1.00      0.91      0.95       100

    accuracy                           0.98       500
   macro avg       0.99      0.96      0.97       500
weighted avg       0.98      0.98      0.98       500

Confusion matrix:
[[400   0]
 [  9  91]]
🏃 View run RF_class_weight_balanced at: http://127.0.0

we can see from this that model type is affecting performance more than class imbalance method, except at the none case where performance crashes, we will pause this part's exploration to focus on model type analysis and comparisn

back again to test knn

In [ ]:

results = []

for strategy_name, strategy in strategies.items():
    run_name = f"knn_{strategy_name}"

    with mlflow.start_run(run_name=run_name):
        if strategy_name == "class_weight_balanced":
            model = KNeighborsClassifier()

            pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("model", model),
                ]
            )
        elif strategy is None:
            model = KNeighborsClassifier()

            pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("model", model),
                ]
            )
        else:
            model = KNeighborsClassifier()

            pipeline = ImbPipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("sampler", strategy),
                    ("model", model),
                ]
            )

        pipeline.fit(X_train, y_train)

        
        y_pred = pipeline.predict(X_val)
        y_prob = None
        if hasattr(pipeline, "predict_proba"):
            y_prob = pipeline.predict_proba(X_val)[:, 1]

        
        accuracy = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, zero_division=0)
        recall = recall_score(y_val, y_pred, zero_division=0)
        f1 = f1_score(y_val, y_pred, zero_division=0)

        report = classification_report(y_val, y_pred, zero_division=0)
        cm = confusion_matrix(y_val, y_pred)

        mlflow.log_param("model", "KNeighborsClassifier")
        mlflow.log_param("imbalance_strategy", strategy_name)
        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("random_state", 10)
        mlflow.log_param("n_numeric_cols", len(numeric_columns))
        mlflow.log_param("n_categorical_cols", len(categorical_cols))


        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)

        report_path = f"classification_report_{strategy_name}.txt"
        with open(report_path, "w") as f:
            f.write(report)

        cm_path = f"confusion_matrix_{strategy_name}.txt"
        with open(cm_path, "w") as f:
            f.write(np.array2string(cm))

        mlflow.log_artifact(report_path)
        mlflow.log_artifact(cm_path)

        # Log model
        mlflow.sklearn.log_model(pipeline, name="model")

        # Store summary
        results.append(
            {
                "strategy": strategy_name,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }
        )

        print(f"\n {strategy_name}:")
        print(report)
        print("Confusion matrix:")
        print(cm)

results_df = pd.DataFrame(results).sort_values(
    by=["f1", "recall", "precision"],
    ascending=False,
)

print("\nFinal comparison")
print(results_df)

Tracking URI: http://127.0.0.1:5000

=== none ===
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       400
           1       1.00      0.87      0.93       100

    accuracy                           0.97       500
   macro avg       0.98      0.94      0.96       500
weighted avg       0.97      0.97      0.97       500

Confusion matrix:
[[400   0]
 [ 13  87]]
🏃 View run knn_none at: http://127.0.0.1:5000/#/experiments/1/runs/435656f074024cb796ae48853e63aed2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

=== class_weight_balanced ===
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       400
           1       1.00      0.87      0.93       100

    accuracy                           0.97       500
   macro avg       0.98      0.94      0.96       500
weighted avg       0.97      0.97      0.97       500

Confusion matrix:
[[400   0]
 [ 13  87]]
🏃 View run knn_class_we